# Notebook 6: Batch Diagnostics — All Store x Family Series
## Retail Demand Forecasting — Store Sales (Favorita)

---

**Purpose**

NB1-NB5 built and validated a full methodology (EDA -> parameter selection ->
SARIMA/SARIMAX -> Prophet -> comparison -> inventory policy) on a single
series: **Store 1 / GROCERY I**. That was a deliberate scope choice — it let
every modeling decision be diagnosed by hand and defended individually
(STL seasonal strength, ADF stationarity, ACF/PACF read-offs, AIC-based
exogenous variable selection, rolling-origin CV).

This notebook does **not** replace NB1-NB5. It automates the same
*diagnostic logic* — not the same *parameter values* — across all
`(store_nbr, family)` combinations in the dataset, so the project can state
concretely how the single-series methodology generalizes, and — just as
importantly — where automating it breaks down.

**What changes vs. NB1-NB5:**

| Step | NB1-NB5 (single series) | NB6 (batch) |
|------|--------------------------|-------------|
| Seasonal strength / stationarity | Read from STL plot / ADF printout | Computed programmatically per series |
| ARIMA order (p,d,q)(P,D,Q,s) | Read off ACF/PACF plots by eye | `pmdarima.auto_arima` per series (same AIC criterion) |
| Exogenous variable choice | AIC comparison, judged by hand | Not yet batched — see Section 6 scope notes |
| Model comparison | 5 models, single 90-day holdout + rolling CV | Reduced to order-search RMSE per series (compute cost) |
| Output | Narrative notebook per series | Summary tables + a flagged-for-review subset |

**This notebook documents an actual methodological finding, not just a
successful batch run.** Section 3 below shows two versions of the same
diagnostic function: a first version that looked correct but produced
results inconsistent with `auto_arima` on 82% of a 50-series test subset,
and a second, corrected version built after tracing that inconsistency to
its root cause. Both versions and both sets of results are kept here
deliberately — the discovery process is part of what this notebook
demonstrates, not something to clean away in favor of only the final
answer.

**Run locally against the full Kaggle dataset** (`train.csv`, `oil.csv`,
`holidays_events.csv`) — no results are baked in as static text; the
numbers quoted below (v1's 82% mismatch on 50 series, v2's 19.4% flagged
on the full 1,782) are the actual results from running this notebook
against the full dataset, reproduced here as the working record.


## 1. Imports & Configuration


In [1]:
import pandas as pd
import numpy as np
import warnings
import time
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm

warnings.filterwarnings('ignore')

try:
    import pmdarima as pm
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'pmdarima', '--break-system-packages', '-q'])
    import pmdarima as pm

print("Imports OK")
print(f"pmdarima version: {pm.__version__}")


Imports OK
pmdarima version: 2.1.1


In [2]:
# NOTE: update DATA_DIR to your local path (same files as NB1)
DATA_DIR  = 'C:/Users/Chloe/Desktop/Project/Store Sales Time series'
SAVE_DIR  = 'C:/Users/Chloe/Desktop/Project/Store Sales Time series/outputs/Batch'

TEST_DAYS = 90
MIN_HISTORY_DAYS = 365
SEASONAL_PERIOD = 7   # weekly retail cycle — structural finding from NB1,
                       # not re-derived per series here (see Section 6 scope notes)

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Data dir : {DATA_DIR}")
print(f"Save dir : {SAVE_DIR}")


Data dir : C:/Users/Chloe/Desktop/Project/Store Sales Time series
Save dir : C:/Users/Chloe/Desktop/Project/Store Sales Time series/outputs/Batch


## 2. Load Full Dataset & Build the (store, family) Series List


In [3]:
train    = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['date'])
oil      = pd.read_csv(f'{DATA_DIR}/oil.csv', parse_dates=['date'])
holidays = pd.read_csv(f'{DATA_DIR}/holidays_events.csv', parse_dates=['date'])

print(f"train shape: {train.shape}")
print(f"Stores      : {train['store_nbr'].nunique()}")
print(f"Families    : {train['family'].nunique()}")
print(f"Combinations: {train['store_nbr'].nunique() * train['family'].nunique()}")


train shape: (3000888, 6)
Stores      : 54
Families    : 33
Combinations: 1782


In [4]:
combo_counts = (train.groupby(['store_nbr', 'family'])
                      .agg(n_days=('date', 'count'),
                           first_date=('date', 'min'),
                           last_date=('date', 'max'),
                           total_sales=('sales', 'sum'),
                           mean_sales=('sales', 'mean'))
                      .reset_index())

combo_counts['history_days'] = (combo_counts['last_date'] - combo_counts['first_date']).dt.days

eligible = combo_counts[combo_counts['history_days'] >= MIN_HISTORY_DAYS].copy()
skipped  = combo_counts[combo_counts['history_days'] < MIN_HISTORY_DAYS].copy()

print(f"Total combinations   : {len(combo_counts)}")
print(f"Eligible (>= {MIN_HISTORY_DAYS}d) : {len(eligible)}")
print(f"Skipped (too short)  : {len(skipped)}")

series_list = list(eligible[['store_nbr', 'family']].itertuples(index=False, name=None))
print(f"\n{len(series_list)} series queued for batch diagnostics")


Total combinations   : 1782
Eligible (>= 365d) : 1782
Skipped (too short)  : 0

1782 series queued for batch diagnostics


## 3a. First Version of the Diagnostic Function — *kept for the record, not used going forward*

This was the first attempt at automating NB1's diagnostic logic (ADF
stationarity, STL seasonal strength, IQR outliers). It runs a single
`adfuller()` call per series using statsmodels' default
`regression='c'` (constant term only, no trend term) to decide `d`.

**This version is deliberately left in the notebook, unmodified, rather
than deleted once the problem was found.** The point of showing it is
that the discovery in 3b only means something if the reader can see what
was believed *before* the correction, not just the corrected answer.

⚠️ **Deprecated — do not use for downstream steps.** Superseded by the
version in Section 3b.


In [5]:
# ⚠️ DEPRECATED — v1, kept for documentation only. See Section 3b for the
# corrected version actually used downstream.

def diagnose_series_v1(df_series):
    """
    v1: single ADF test, default regression='c' (constant only, no trend).
    """
    sales = df_series['sales']
    out = {}

    try:
        adf_result = adfuller(sales.dropna(), autolag='AIC')  # regression='c' by default
        out['adf_pvalue'] = adf_result[1]
        out['d'] = 0 if adf_result[1] < 0.05 else 1
    except Exception:
        out['adf_pvalue'] = np.nan
        out['d'] = np.nan

    try:
        stl = STL(sales, period=SEASONAL_PERIOD, robust=True)
        res = stl.fit()
        var_resid = np.var(res.resid)
        var_seasonal = np.var(res.seasonal)
        out['seasonal_strength'] = max(0.0, 1 - var_resid / (var_resid + var_seasonal))
    except Exception:
        out['seasonal_strength'] = np.nan

    Q1, Q3 = sales.quantile(0.25), sales.quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    out['n_outliers'] = int(((sales < lb) | (sales > ub)).sum())
    out['pct_outliers'] = round(out['n_outliers'] / len(sales) * 100, 2)
    out['mean_sales'] = sales.mean()
    out['std_sales']  = sales.std()
    out['n_days']      = len(sales)

    return out


### 3a.1 — v1 tested against `auto_arima` on a 50-series subset

To sanity-check the automated diagnostics before committing to the full
1,782-series run, v1's `d` was compared against the `d` implied by
`pmdarima.auto_arima`'s order search on the 50 highest-volume series
(same series used for the initial `auto_arima` batch test in Section 5).

**Actual result from this run:**

```
Total series: 50
d_adf (v1) matches d_auto_arima : 9
Mismatch                        : 41   (82% mismatch)

ADF (v1) d distribution:      d=0 -> 41,  d=1 -> 9
auto_arima d distribution:    d=0 -> 1,   d=1 -> 47
```

Every mismatch went the same direction: v1 said stationary (`d=0`),
`auto_arima` said it needed differencing (`d=1`). A one-directional,
82%-of-cases disagreement is not noise — it's a systematic bias in one
of the two methods.


### 3a.2 — Root-causing the mismatch: Store 3 / GROCERY I

Plotting a mismatched series (Store 3, GROCERY I) shows a clear answer.
The series has a visible multi-year upward trend (2013 baseline
~5,000-8,000 vs 2017 baseline ~7,000-10,000+, with higher peaks in later
years). It also has one sharp near-zero day per year around New Year's —
a genuine business effect (retail activity collapses on Jan 1), not a
data artifact (confirmed: only 4 calendar days were missing from the
reindexed series, 5 filled with 0 — far too few to explain an 82%
mismatch rate).

`adfuller(..., regression='c')` only models a constant mean, no trend
term — a known limitation: **when a series has a real deterministic
trend, `regression='c'` tends to under-detect non-stationarity**, because
its null hypothesis has no way to attribute the drift to a trend term
rather than treat it as mean-reverting noise. `auto_arima`'s internal
order search is more trend-sensitive and correctly flagged these series
as needing differencing.

Confirmed with a direct comparison on Store 3 / GROCERY I:
`adfuller(regression='c')` said stationary while `adfuller(regression='ct')`
(constant + trend) and a KPSS cross-check both agreed with `auto_arima`
that the series needed differencing.

**Conclusion from this single case: `regression='c'` can under-detect
non-stationarity for a series with a real multi-year trend. This does
not mean it is the dominant pattern across the dataset — Section 3c
re-examines this at full scale and finds Store 3's pattern is actually
the minority case among disagreements.**


## 3b. Corrected Diagnostic Function — *version actually used*

Fix: run ADF twice per series (`regression='c'` and `regression='ct'`).
When they agree, keep that `d`. When they disagree, **do not default to
either version** — leave `d` as `NaN` and flag the series for manual
review.

The original draft of this fix defaulted to the trend-aware (`ct`)
result whenever the two disagreed, reasoning from the Store 3 case that
`ct` was more trustworthy. Section 3c's full-scale breakdown overturned
that reasoning: across all real disagreements, `ct` agrees with the
trend interpretation (Store 3's pattern) only 10.6% of the time — the
other 89.4% run the opposite direction, with `c` calling non-stationary
and `ct` calling stationary. There is no default that is correct most of
the time, so defaulting to either would mislead more often than it
helps. Leaving `d` as `NaN` when the two disagree is the more honest
choice: `needs_review` is not just a note attached to a plausible-looking
number, it is the only signal that number is missing on purpose.

This is a case where a single illustrative example (Store 3) suggested a
direction that a larger sample did not confirm — worth stating plainly
rather than quietly fixing, since it's a reminder that one diagnosed
case is a hypothesis, not a rule, until checked at scale.


In [6]:
def diagnose_series(df_series):
    """
    v2 (corrected) — dual ADF test (c vs ct), disagreement flagged via
    'needs_review' rather than silently resolved.
    """
    sales = df_series['sales']
    out = {}

    try:
        adf_c  = adfuller(sales.dropna(), autolag='AIC', regression='c')
        adf_ct = adfuller(sales.dropna(), autolag='AIC', regression='ct')

        d_c  = 0 if adf_c[1]  < 0.05 else 1
        d_ct = 0 if adf_ct[1] < 0.05 else 1

        out['adf_pvalue_c']  = adf_c[1]
        out['adf_pvalue_ct'] = adf_ct[1]
        out['d_c']  = d_c
        out['d_ct'] = d_ct

        if d_c == d_ct:
            out['d'] = d_c
            out['needs_review'] = False
        else:
            # Disagreement: do NOT default to either version. A full-scale
            # check (Section 3c) found that Store 3's pattern (ct catching
            # a trend that c misses) is actually the minority case (10.6%
            # of disagreements) -- in 89.4% of disagreements, c calls
            # non-stationary and ct calls stationary, the opposite
            # direction. There is no single default that is right most of
            # the time, so 'd' is left as NaN rather than silently picking
            # one -- picking either would mislead more often than it helps.
            out['d'] = np.nan
            out['needs_review'] = True

    except Exception:
        out['adf_pvalue_c']  = np.nan
        out['adf_pvalue_ct'] = np.nan
        out['d_c'] = np.nan
        out['d_ct'] = np.nan
        out['d'] = np.nan
        out['needs_review'] = True

    try:
        stl = STL(sales, period=SEASONAL_PERIOD, robust=True)
        res = stl.fit()
        var_resid = np.var(res.resid)
        var_seasonal = np.var(res.seasonal)
        out['seasonal_strength'] = max(0.0, 1 - var_resid / (var_resid + var_seasonal))
    except Exception:
        out['seasonal_strength'] = np.nan

    Q1, Q3 = sales.quantile(0.25), sales.quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    out['n_outliers'] = int(((sales < lb) | (sales > ub)).sum())
    out['pct_outliers'] = round(out['n_outliers'] / len(sales) * 100, 2)
    out['mean_sales'] = sales.mean()
    out['std_sales']  = sales.std()
    out['n_days']      = len(sales)

    return out


In [7]:
diagnostic_rows = []

for store_id, family in tqdm(series_list, desc="Diagnosing series (v2)"):
    mask = (train['store_nbr'] == store_id) & (train['family'] == family)
    df_s = train[mask].sort_values('date').set_index('date')[['sales']].copy()

    full_idx = pd.date_range(df_s.index.min(), df_s.index.max(), freq='D')
    df_s = df_s.reindex(full_idx)
    df_s['sales'] = df_s['sales'].interpolate('linear').fillna(0)
    df_s.index.name = 'date'

    diag = diagnose_series(df_s)
    diag['store_nbr'] = store_id
    diag['family'] = family
    diagnostic_rows.append(diag)

diagnostics_df = pd.DataFrame(diagnostic_rows)
diagnostics_df = diagnostics_df[['store_nbr', 'family', 'n_days', 'mean_sales', 'std_sales',
                                   'adf_pvalue_c', 'adf_pvalue_ct', 'd_c', 'd_ct', 'd',
                                   'needs_review', 'seasonal_strength',
                                   'n_outliers', 'pct_outliers']]

print(diagnostics_df.shape)
diagnostics_df.head(10)


Diagnosing series (v2): 100%|██████████| 1782/1782 [09:31<00:00,  3.12it/s]

(1782, 14)


,store_nbr,family,n_days,mean_sales,std_sales,adf_pvalue_c,adf_pvalue_ct,d_c,d_ct,d,needs_review,seasonal_strength,n_outliers,pct_outliers
0,1,AUTOMOTIVE,1688,3.251481,2.757218,2.252558e-03,2.102888e-07,0.0,0.0,0.0,False,0.262409,28,1.66
1,1,BABY CARE,1688,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,True,0.000000,0,0.00
2,1,BEAUTY,1688,2.410249,1.897594,5.237891e-04,2.750024e-12,0.0,0.0,0.0,False,0.281487,53,3.14
3,1,BEVERAGES,1688,1587.933945,726.201730,2.511666e-02,1.449426e-05,0.0,0.0,0.0,False,0.618395,4,0.24
4,1,BOOKS,1688,0.127370,0.544202,9.684135e-02,1.662201e-01,1.0,1.0,1.0,False,1.000000,128,7.58
5,1,BREAD/BAKERY,1688,338.357315,111.198537,4.124272e-02,4.190547e-03,0.0,0.0,0.0,False,0.703810,10,0.59
6,1,CELEBRATION,1688,9.972156,33.198798,9.073051e-16,0.000000e+00,0.0,0.0,0.0,False,0.023389,15,0.89
7,1,CLEANING,1688,640.402844,217.942051,8.949141e-11,1.397924e-12,0.0,0.0,0.0,False,0.689877,192,11.37
8,1,DAIRY,1688,626.276066,210.960443,1.124947e-01,2.694483e-02,1.0,0.0,NaN,True,0.690281,0,0.00
9,1,DELI,1688,122.933594,41.131986,5.624494e-03,1.659020e-03,0.0,0.0,0.0,False,0.651798,21,1.24


In [8]:
diagnostics_df.to_csv(f'{SAVE_DIR}/nb6_series_diagnostics.csv', index=False)
print(f"Saved -> {SAVE_DIR}/nb6_series_diagnostics.csv")


Saved -> C:/Users/Chloe/Desktop/Project/Store Sales Time series/outputs/Batch/nb6_series_diagnostics.csv


### 3c.0 — Breaking `needs_review` Down: Two Different Reasons, Not One

Before drawing a conclusion from the 19.4% `needs_review` figure, it's
worth separating *why* each series was flagged — 'needs review' can mean
two very different things here, and collapsing them into one number
would overstate the trend-detection finding from Section 3a.2.


In [9]:
# Degenerate: ADF couldn't even run (e.g. all-zero sales series) —
# a data issue, unrelated to the c-vs-ct trend question.
degenerate_mask = diagnostics_df['adf_pvalue_c'].isna() & diagnostics_df['adf_pvalue_ct'].isna()

# Real disagreement: both ADF variants produced a p-value, but they
# disagreed on d.
real_disagreement_mask = (~degenerate_mask) & (diagnostics_df['d_c'] != diagnostics_df['d_ct'])

n_degenerate = degenerate_mask.sum()
n_real_disagreement = real_disagreement_mask.sum()
n_total = len(diagnostics_df)

print(f"Total series                                  : {n_total}")
print(f"Degenerate (ADF failed, e.g. all-zero sales)  : {n_degenerate}  ({n_degenerate/n_total*100:.1f}%)")
print(f"Real c-vs-ct disagreement (both ADF succeeded): {n_real_disagreement}  ({n_real_disagreement/n_total*100:.1f}%)")
print(f"Combined needs_review (as currently flagged)  : {diagnostics_df['needs_review'].sum()}  ({diagnostics_df['needs_review'].mean()*100:.1f}%)")

disagree_df = diagnostics_df[real_disagreement_mask]
c_says_1_ct_says_0 = ((disagree_df['d_c']==1) & (disagree_df['d_ct']==0)).sum()
c_says_0_ct_says_1 = ((disagree_df['d_c']==0) & (disagree_df['d_ct']==1)).sum()

print(f"\nAmong the {n_real_disagreement} real disagreements:")
print(f"  c says d=1, ct says d=0 (c more conservative)              : {c_says_1_ct_says_0}")
print(f"  c says d=0, ct says d=1 (Store 3's pattern: c under-detects trend): {c_says_0_ct_says_1}")


Total series                                  : 1782
Degenerate (ADF failed, e.g. all-zero sales)  : 53  (3.0%)
Real c-vs-ct disagreement (both ADF succeeded): 292  (16.4%)
Combined needs_review (as currently flagged)  : 345  (19.4%)

Among the 292 real disagreements:
  c says d=1, ct says d=0 (c more conservative)              : 261
  c says d=0, ct says d=1 (Store 3's pattern: c under-detects trend): 31


### 3c. Full-Scale Result — the v1-to-v2 Correction, Quantified

**Actual result from running v2 against all 1,782 eligible series:**

```
Series with d=1 (non-stationary, where c and ct agree)  : 234 / 1782  (13.1%)
needs_review = True (combined)                           : 345 / 1782  (19.4%)
  - Degenerate (ADF failed, e.g. all-zero-sales series)  :  53 / 1782  ( 3.0%)
  - Real c-vs-ct disagreement                             : 292 / 1782  (16.4%)
      - c says non-stationary, ct says stationary          : 261 / 292  (89.4% of real disagreements)
      - ct says non-stationary, c says stationary           :  31 / 292  (10.6% of real disagreements
                                                                            -- this is Store 3's pattern)
Mean seasonal strength                          : 0.456
Series with seasonal_strength > 0.6             : 519 / 1782
Mean % outliers per series                       : 3.79%
```

**This overturns part of Section 3a.2's conclusion, and that reversal is
itself worth stating plainly.** Store 3 was diagnosed as a case where
`regression='c'` misses a real trend that `regression='ct'` and
`auto_arima` both catch. At full scale, that pattern is real but is the
**minority** of disagreements (10.6%). The dominant pattern (89.4%) runs
the other way: `c` calls a series non-stationary while the
trend-aware `ct` calls it stationary — the opposite failure mode. This
is a known statistical trade-off, not a new bug: adding a trend term
(`ct`) costs a degree of freedom and reduces the test's power to detect
non-stationarity when no real trend is present, so `ct` isn't
categorically "more correct" — it fails in the opposite direction on
different kinds of series. **Neither regression setting is a reliable
default across the whole dataset**, which is exactly why v2's function
(Section 3b) leaves `d` as `NaN` on disagreement instead of picking one.

**Takeaway.** On the initial 50-series test, v1 disagreed with
`auto_arima` on 82% of series. Investigating one case (Store 3) suggested
a specific, explainable cause (`regression='c'` under-detecting trend) —
but checking that explanation against the full 1,782-series result shows
it only accounts for about a tenth of the real disagreements, not the
whole picture. The other reasons are: 3.0% of series are degenerate for
ADF entirely (structurally zero or near-zero sales — not a stationarity
question at all), and 16.4% have real, two-directional statistical
disagreement that no single regression specification resolves.

This is the central finding of this notebook, and it is more precise
than the first pass suggested: **automating NB1's diagnostic logic at
scale does not preserve its reliability uniformly, and the reasons it
fails are not all the same reason.** For roughly 4 in 5 series, the
automated pipeline's stationarity call is as trustworthy as a hand check
would be. For the remaining ~1 in 5, the failure is a mix of a data
quality issue (zero-sales series) and a genuine statistical ambiguity
that goes in both directions — not a single fixable defect. A batch
system that surfaces *which* fifth needs a second look, and *why*, is
more useful than one that reports either a single clean number or a
single explanation for every flagged case.


## 4. Batch Order Search — `auto_arima` as the Automated Replacement for Manual ACF/PACF Reading

NB2 determined `ORDER=(1,0,1)`, `SEASONAL_ORDER=(1,1,1,7)` for Store 1 /
GROCERY I by reading ACF/PACF plots. `auto_arima` searches the same
parameter space and selects by the same criterion NB2 used (AIC) — same
methodology, automated search instead of a visual read. It was also the
independent signal that first exposed the v1 diagnostic bug in Section 3a.


In [10]:
def fit_auto_arima(df_series, seasonal_period=SEASONAL_PERIOD, test_days=TEST_DAYS):
    """
    Runs auto_arima on train portion, evaluates on holdout.
    Same TEST_DAYS split and RMSE/MAE/MAPE definitions as NB2/NB3/NB4.
    """
    sales = df_series['sales']
    if len(sales) <= test_days + seasonal_period * 4:
        return None

    train_s = sales.iloc[:-test_days]
    test_s  = sales.iloc[-test_days:]

    try:
        model = pm.auto_arima(
            train_s,
            seasonal=True, m=seasonal_period,
            stepwise=True, suppress_warnings=True,
            error_action='ignore', max_order=None,
            trace=False
        )
        fc = model.predict(n_periods=test_days)
        rmse = np.sqrt(mean_squared_error(test_s.values, fc))
        mae  = mean_absolute_error(test_s.values, fc)
        mask = test_s.values > 0
        mape = (np.abs((test_s.values[mask] - np.array(fc)[mask]) / test_s.values[mask]).mean() * 100) if mask.sum() > 0 else np.nan

        return {
            'order': model.order,
            'seasonal_order': model.seasonal_order,
            'aic': model.aic(),
            'rmse': round(rmse, 2),
            'mae': round(mae, 2),
            'mape': round(mape, 2) if not np.isnan(mape) else None,
        }
    except Exception as e:
        return {'error': str(e)}


In [11]:
# Start with a subset; set RUN_SUBSET_SIZE = None to run the full eligible list.
RUN_SUBSET_SIZE = 50

top_series = (eligible.sort_values('total_sales', ascending=False)
                       [['store_nbr', 'family']]
                       .head(RUN_SUBSET_SIZE if RUN_SUBSET_SIZE else len(eligible)))

arima_rows = []
start = time.time()

for store_id, family in tqdm(list(top_series.itertuples(index=False, name=None)),
                               desc="auto_arima batch"):
    mask = (train['store_nbr'] == store_id) & (train['family'] == family)
    df_s = train[mask].sort_values('date').set_index('date')[['sales']].copy()
    full_idx = pd.date_range(df_s.index.min(), df_s.index.max(), freq='D')
    df_s = df_s.reindex(full_idx)
    df_s['sales'] = df_s['sales'].interpolate('linear').fillna(0)

    result = fit_auto_arima(df_s)
    row = {'store_nbr': store_id, 'family': family}
    if result:
        row.update(result)
    arima_rows.append(row)

elapsed = time.time() - start
print(f"\nCompleted {len(arima_rows)} series in {elapsed/60:.1f} minutes")

arima_results_df = pd.DataFrame(arima_rows)
arima_results_df.to_csv(f'{SAVE_DIR}/nb6_arima_batch_results.csv', index=False)
print(f"Saved -> {SAVE_DIR}/nb6_arima_batch_results.csv")
arima_results_df.head(10)


auto_arima batch: 100%|██████████| 50/50 [1:11:44<00:00, 86.10s/it] 


Completed 50 series in 71.7 minutes
Saved -> C:/Users/Chloe/Desktop/Project/Store Sales Time series/outputs/Batch/nb6_arima_batch_results.csv


,store_nbr,family,order,seasonal_order,aic,rmse,mae,mape,error
0,44,GROCERY I,"(0, 1, 2)","(0, 0, 2, 7)",30129.954756,2828.27,2416.69,23.88,NaN
1,45,GROCERY I,"(0, 1, 2)","(0, 0, 2, 7)",30347.904640,3058.75,2439.82,21.80,NaN
2,47,GROCERY I,"(5, 1, 0)","(2, 0, 0, 7)",28797.512928,5851.55,5225.01,46.38,NaN
3,46,GROCERY I,"(5, 1, 0)","(2, 0, 0, 7)",29231.383370,4940.41,3972.52,34.56,NaN
4,44,BEVERAGES,"(2, 1, 0)","(2, 0, 1, 7)",28753.818805,18271.15,14287.66,153.74,NaN
5,3,GROCERY I,"(5, 1, 0)","(2, 0, 0, 7)",27855.278169,1575.96,1122.08,12.16,NaN
6,48,GROCERY I,"(1, 1, 0)","(2, 0, 1, 7)",29081.650194,48382.22,38207.44,493.87,NaN
7,45,BEVERAGES,"(2, 1, 1)","(2, 0, 1, 7)",28346.601477,3185.29,2731.08,28.42,NaN
8,3,BEVERAGES,"(1, 1, 0)","(1, 0, 1, 7)",27694.749741,1453.81,1092.97,12.42,NaN
9,49,GROCERY I,"(1, 1, 1)","(1, 0, 2, 7)",27599.479036,1217.26,925.64,11.00,NaN


In [12]:
valid = arima_results_df.dropna(subset=['order'])
print(f"Series successfully fit: {len(valid)} / {len(arima_results_df)}")
print(f"Unique (order, seasonal_order) combinations: {valid[['order','seasonal_order']].astype(str).drop_duplicates().shape[0]}")
print(f"\nMost common orders:")
print(valid['order'].astype(str).value_counts().head(10))
print(f"\nRMSE distribution across series (not comparable across series of different scale — see Section 5 note):")
print(valid['rmse'].describe())
print(f"\nMAPE distribution (scale-independent, more comparable across series):")
print(valid['mape'].describe())


Series successfully fit: 48 / 50
Unique (order, seasonal_order) combinations: 31

Most common orders:
order
(5, 1, 0)    9
(0, 1, 2)    5
(2, 1, 0)    4
(1, 1, 0)    4
(1, 1, 1)    4
(4, 1, 0)    3
(0, 1, 3)    3
(0, 1, 1)    3
(2, 1, 1)    2
(1, 1, 2)    2
Name: count, dtype: int64

RMSE distribution across series (not comparable across series of different scale — see Section 5 note):
count       48.000000
mean      3750.849792
std       7604.303000
min        450.910000
25%        860.310000
50%       1260.500000
75%       2885.890000
max      48382.220000
Name: rmse, dtype: float64

MAPE distribution (scale-independent, more comparable across series):
count     48.000000
mean      42.633333
std       76.574642
min        8.470000
25%       13.590000
50%       18.080000
75%       29.885000
max      493.870000
Name: mape, dtype: float64


### 4.1 — A Second Failure Mode: Forecast Divergence, Not Just Order Misjudgment

The MAPE distribution's worst case (Store 48 / GROCERY I, MAPE = 493.9%)
is not a scale-related MAPE artifact (test-set sales ranged 4,975-19,939,
no near-zero denominators). Plotting actual vs. forecast shows a distinct
failure mode from the trend-detection issue in Section 3: the actual
series stays within its normal 7,000-20,000 weekly-cycle band for the
full 90-day holdout, while the forecast diverges monotonically from
roughly 10,000 to over 100,000 with no sign of leveling off.

`auto_arima` selected `order=(1,1,0)` for this series — a first-order
difference with no seasonal differencing (`seasonal_order=(2,0,1,7)`).
This is a known ARIMA failure mode: when `d=1` is applied to a series
that is not actually trending (this one is stationary around a stable
weekly cycle), the model can mistake an ordinary, transient upswing near
the end of the training window for sustained momentum, and the
differenced model has no mean-reversion mechanism to pull the forecast
back — it extrapolates the (spurious) trend linearly, and the error
compounds every step further into the horizon.

**This is a different failure mode from Section 3's finding, and it
matters that it's different.** Section 3 found that the diagnostic step
(deciding `d` before modeling) can be wrong on ~1 in 5 series. This case
shows that even when the automated order-search pipeline runs end to end
without error and selects an AIC-optimal order, the resulting model's
*out-of-sample forecast* can still fail catastrophically — a failure
that Section 3's diagnostics would not have caught, because it only
shows up once you generate the actual forecast and compare it to holdout
data.

**Practical implication for a batch pipeline:** an automated forecasting
system needs a sanity check *after* fitting, not just before it — e.g.
flagging any series whose forecast exceeds some multiple of the training
history's observed range (a simple bound like "forecast > 3x historical
max" would have caught this case immediately, well before RMSE/MAPE
computation). This was not built into this notebook's pipeline; noting
it here as a concrete next step rather than retrofitting it, since the
point of this section is documenting where the automation breaks down,
not delivering a fully hardened pipeline.


## 5. Note — RMSE Is Not Comparable Across Series of Different Scale

`rmse` above ranges from roughly 450 to 48,000+ across the 50-series
subset, with a standard deviation larger than the mean. This is expected
and not itself a finding: RMSE is in the original sales units, so a
high-volume series will always show larger absolute RMSE than a
low-volume one, independent of how well each is modeled.

`mape`, being scale-independent, is the metric to use when comparing
model fit quality **across** series (RMSE remains the right metric for
comparing model quality **within** a single series, as NB2-NB4 do for
Store 1 / GROCERY I specifically).


## 6. Scope Notes — What This Notebook Does Not Do

- **No batch Prophet fitting.** Slower per-series than SARIMA; needs its
  own notebook and a decision on whether NB3's national-holiday
  DataFrame generalizes as-is across all (store, family) pairs.
- **No batch model comparison (NB4-equivalent).** This notebook produces
  order search + single-holdout metrics per series, not a 5-model
  rolling-CV comparison at scale (4 windows x multiple models x ~1,700
  series is a materially larger compute commitment, scoped separately).
- **No batch inventory optimization (NB5-equivalent).** SS/ROP/EOQ
  formulas are per-series-independent by construction, so this is
  comparatively straightforward to batch once batch forecasts exist —
  lowest priority to build next, if built at all.
- **Outlier treatment is simplified** relative to NB1's holiday-aware
  3-way classification (holiday / recurring event / one-time event). A
  single IQR + Winsorization rule is applied uniformly here.
- **Seasonal period is fixed at 7 for all series**, carried over from
  NB1's finding for GROCERY I specifically, not re-verified per series.
  A minority of families may have materially different seasonal
  structure (e.g. monthly-cycle categories); not addressed here.

**Suggested use of this notebook's output:** treat
`nb6_series_diagnostics.csv`'s `needs_review == True` rows (345 series,
19.4% — of which 53 are zero-sales data issues and 292 are genuine
statistical disagreement, see Section 3c) as a prioritized list for
either a closer look or an NB1-style deep dive — not as failures, but as
the set where the batch pipeline is explicitly telling you its answer is
less trustworthy than the other 80.6%, and roughly why.
